qsub run_exp5_variable_baseline_ClimaX.pbs

# CMIP variable prediction — baseline ClimaX

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import json
from pathlib import Path
import pickle
import math
import os
import time

EXPERIENCE 5 HYPERPARAMETERS     :

In [ ]:
setup_name = "baseline_ClimaX"

In [ ]:
num_sample = 2500000
variable = "pr"
val_fraction = 0.05
test_fraction = 0.15

**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

# Split off the specified variable as a dedicated label while keeping sample/grid-point alignment.
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

label_variable_by_climate = {}
features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )
    label_variable_by_climate[c] = X_full[:, variable_col_start:variable_col_end].copy()
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]

label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Feature variables used downstream (without {variable}): {selected_variables}")
print(f"label_{variable} shape (all samples x grid points): {label_variable.shape}")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
climax_train_climates = ["historical"]

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- split train/val/test first, then fit scalers only on the historical train split

# Baseline ClimaX — Fine-tuning a Climate Foundation Model

This notebook implements a ClimaX-based baseline for predicting the precipitation field (`pr`)
from the multivariate CMIP6 input patches.

**Architecture**:  
ClimaX-style Vision Transformer that tokenizes each (variable, spatial_position) pair
independently. The Transformer output is compressed to a 64-dim global latent via
progressive compression (1024→256→64) and attention pooling over spatial positions.
A CERA-style MLP head (5 FC layers, 128 units, LeakyReLU) predicts all output grid points
from this 64-dim latent.

**Training protocol** (identical to other exp-5 baselines for split/normalization):  
- **Frozen backbone**: token embeddings, channel (variable) embeddings, and all 8 pretrained ClimaX Transformer blocks + final LayerNorm are loaded from tungnd/climax (5.625deg.ckpt) and kept entirely frozen throughout training (requires_grad=False) — no gradient ever flows through the backbone.
- **Trained parameters**: only the prediction head — progressive compression, attention pooling, and the CERA-style predictor (~450K params out of ~100M total). Positional embeddings are not a learned model parameter: they are precomputed once per patch by interpolating the pretrained pos_embed grid and passed into the model as a fixed input.
- Input variables : all CMIP6 variables loaded from samples, **excluding** the target (`pr`).
- Target         : `pr` field at all patch grid points.
- Train split    : historical train only.
- Validation     : historical val (early stopping).
- Test sets      : historical test + all SSP test sets.
- Normalization  : identical pipeline (variable-wise, log1p for `pr`, fit on historical train).

**Output format**:  
Same `quality_payload` dict and pkl files as the other exp-5 baselines.
Latent representations (64-dim, comparable to CERA's `ae_latent_dim=64`) are also saved.

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Live progress log (separate from notebook cell outputs) ──────────────────
# nbclient only writes cell outputs to the executed .ipynb at the very end
# (or on error) -- print() inside cells is NOT streamed to the PBS job's
# stdout/stderr while the job is running. This file is written directly from
# the kernel process, so `tail -f` on it shows real-time progress regardless.
_progress_log_path = Path("/glade/u/home/tsalin/CMIP/logs/exp5_variable_baseline_ClimaX_progress.log")
_progress_log_path.parent.mkdir(parents=True, exist_ok=True)


def log_progress(msg: str) -> None:
    with open(_progress_log_path, "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}\n")


log_progress(f"Notebook (re)started. Device: {device}")

Train/test split and rigorous normalization

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices for each climate before any standardization."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices


# Build splits on raw, unstandardized data first.
ae_split_indices = build_split_indices(
    features_by_climate,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimization: truncate eval-only climates to their test split ───────
_eval_only_climates = [c for c in climate_order if c not in climax_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        _X_sub = features_by_climate[c][_idx]
        features_by_climate[c] = _X_sub
        _y_sub = label_variable_by_climate[c][_idx]
        label_variable_by_climate[c] = _y_sub
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt: {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx, _X_sub, _y_sub
del _eval_only_climates
# ── End RAM optimization ──────────────────────────────────────────────────────

# -----------------------------------------------------------------------------
# Rigorous variable-wise standardization
# -----------------------------------------------------------------------------
# Inputs:
#   - fit normalization statistics only on historical train samples;
#   - estimate one mean/std per physical input variable using all grid points of
#     the patch together;
#   - apply these variable-wise statistics to all climates and all splits.
#
# Label:
#   - fit a separate normalization for the target variable only on historical
#     train labels;
#   - estimate one mean/std over all target grid points together;
#   - apply it to all climates and all splits without refit.
#
# This avoids train/test leakage while preserving the physical spatial structure
# of each variable.

reference_climate = "historical"
if reference_climate not in features_by_climate:
    raise KeyError(f"Reference climate {reference_climate!r} not found in features_by_climate.")

hist_train_idx = ae_split_indices[reference_climate]["train"]

X_hist_train_raw = np.asarray(features_by_climate[reference_climate][hist_train_idx], dtype=np.float32)
y_hist_train_raw = np.asarray(label_variable_by_climate[reference_climate][hist_train_idx], dtype=np.float32)

n_input_variables = len(selected_variables)
expected_input_dim = n_input_variables * grid_points_per_patch
if X_hist_train_raw.shape[1] != expected_input_dim:
    raise ValueError(
        f"Unexpected input dimension: got {X_hist_train_raw.shape[1]}, "
        f"expected {expected_input_dim} = {n_input_variables} variables x {grid_points_per_patch} points."
    )
if y_hist_train_raw.shape[1] != grid_points_per_patch:
    raise ValueError(
        f"Unexpected label dimension: got {y_hist_train_raw.shape[1]}, expected {grid_points_per_patch}."
    )

input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds = np.ones(n_input_variables, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(
        var_idx * grid_points_per_patch,
        (var_idx + 1) * grid_points_per_patch,
    )
    values = X_hist_train_raw[:, cols_for_var].reshape(-1)
    values = values[np.isfinite(values)]

    if values.size == 0:
        raise ValueError(f"No finite historical train values found for input variable {var_name!r}.")

    if var_name == "pr":
        values = np.log1p(values * 86400)

    mu = float(np.mean(values))
    sigma = float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx] = sigma

label_values = y_hist_train_raw.reshape(-1)
label_values = label_values[np.isfinite(label_values)]
if label_values.size == 0:
    raise ValueError(f"No finite historical train values found for label variable {variable!r}.")

if variable == "pr":
    label_values = np.log1p(label_values * 86400)

label_mean = float(np.mean(label_values))
label_std = float(np.std(label_values))
if not np.isfinite(label_std) or label_std <= 0:
    label_std = 1.0


def standardize_input_variables(X_raw):
    """Apply historical-train variable-wise normalization to the input variables."""
    X_raw = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx])
            / input_variable_stds[var_idx]
        ).astype(np.float32)

    return X_scaled


def standardize_label_variable(y_raw):
    """Apply target-variable standardization to all grid points of the label."""
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if variable == "pr":
        y_raw = np.log1p(y_raw * 86400)
    return ((y_raw - label_mean) / label_std).astype(np.float32)


def denormalize_label_variable(y_scaled):
    """Convert standardized target-variable arrays back to physical units (mm/day for pr)."""
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    result = y_scaled * label_std + label_mean
    if variable == "pr":
        result = np.expm1(result)
    return result.astype(np.float32)


scaled_features_by_climate = {}
scaled_label_variable_by_climate = {}

for climate in climate_order:
    X_raw = np.asarray(features_by_climate[climate], dtype=np.float32)
    y_raw = np.asarray(label_variable_by_climate[climate], dtype=np.float32)

    scaled_features_by_climate[climate] = standardize_input_variables(X_raw)
    scaled_label_variable_by_climate[climate] = standardize_label_variable(y_raw)

# From this point onward, label_variable_by_climate contains scaled targets.
label_variable_by_climate = scaled_label_variable_by_climate
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

input_standardization_summary_df = pd.DataFrame({
    "variable": selected_variables,
    "mean_fit_on_historical_train_all_points": input_variable_means,
    "std_fit_on_historical_train_all_points": input_variable_stds,
})
label_standardization_summary_df = pd.DataFrame({
    "variable": [variable],
    "mean_fit_on_historical_train_all_points": [label_mean],
    "std_fit_on_historical_train_all_points": [label_std],
})

print("\u2713 Built train/val/test splits before standardization.")
print(f"\u2713 Input normalization fitted on {reference_climate} train only, variable-wise over all patch points.")
print(f"\u2713 Label normalization fitted on {reference_climate} train only, over all target grid points.")
print("\u2713 Applied the same fitted normalization statistics to all climates without refit.")
print("Scaled input shapes:", {c: scaled_features_by_climate[c].shape for c in climate_order})
print("Scaled label shapes:", {c: scaled_label_variable_by_climate[c].shape for c in climate_order})

display(input_standardization_summary_df)
display(label_standardization_summary_df)

In [ ]:
# RAM Reduction
del features_by_climate_full
del features_by_climate

In [ ]:
# ── ClimaX ERA5 variable list (5.625 deg checkpoint -- 48 variables) ──────────
# Order is fixed by the checkpoint; determines channel_embed and token_embeds indices.
CLIMAX_VARS = [
    "land_sea_mask", "orography", "lattitude",                               # 0-2
    "2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind",  # 3-5
    "toa_incident_solar_radiation", "total_precipitation",                   # 6-7
    "geopotential_50",   "geopotential_250",  "geopotential_500",            # 8-10
    "geopotential_600",  "geopotential_700",  "geopotential_850",            # 11-13
    "geopotential_925",                                                       # 14
    "u_component_of_wind_50",  "u_component_of_wind_250", "u_component_of_wind_500",   # 15-17
    "u_component_of_wind_600", "u_component_of_wind_700", "u_component_of_wind_850",   # 18-20
    "u_component_of_wind_925",                                                # 21
    "v_component_of_wind_50",  "v_component_of_wind_250", "v_component_of_wind_500",   # 22-24
    "v_component_of_wind_600", "v_component_of_wind_700", "v_component_of_wind_850",   # 25-27
    "v_component_of_wind_925",                                                # 28
    "temperature_50", "temperature_250", "temperature_500",                  # 29-31
    "temperature_600", "temperature_700", "temperature_850", "temperature_925",  # 32-35
    "specific_humidity_50",  "specific_humidity_250", "specific_humidity_500",   # 36-38
    "specific_humidity_600", "specific_humidity_700", "specific_humidity_850",   # 39-41
    "specific_humidity_925",                                                  # 42
    "vorticity_500", "vorticity_850",                                        # 43-44
    "mean_sea_level_pressure", "10m_wind_speed", "total_cloud_cover",        # 45-47
]
assert len(CLIMAX_VARS) == 48

# CMIP6 variable -> index in the ClimaX 48-var list.
# Variables without a direct ERA5 equivalent are dropped (huss, wap500, rsds).
CMIP6_TO_CLIMAX_IDX = {
    "tas":      3,   # 2m_temperature
    "ta850":   34,   # temperature_850
    "ta500":   31,   # temperature_500
    "hus850":  41,   # specific_humidity_850
    "hus500":  38,   # specific_humidity_500
    "ua850":   20,   # u_component_of_wind_850
    "va850":   27,   # v_component_of_wind_850
    "ua500":   17,   # u_component_of_wind_500
    "va500":   24,   # v_component_of_wind_500
    "zg500":   10,   # geopotential_500
    "psl":     45,   # mean_sea_level_pressure
    "sfcWind": 46,   # 10m_wind_speed
}

matched_variables   = [v for v in selected_variables if v in CMIP6_TO_CLIMAX_IDX]
matched_climax_idxs = [CMIP6_TO_CLIMAX_IDX[v] for v in matched_variables]
dropped_variables   = [v for v in selected_variables if v not in CMIP6_TO_CLIMAX_IDX]
local_idxs_matched  = [selected_variables.index(v) for v in matched_variables]
n_matched           = len(matched_variables)

print(f"Matched  ({n_matched}): {matched_variables}")
print(f"Dropped  ({len(dropped_variables)}): {dropped_variables}")


def _subset_matched(X_flat):
    # Extract only matched-variable columns from flat feature array [n, V*P]
    return np.concatenate(
        [X_flat[:, i * grid_points_per_patch : (i + 1) * grid_points_per_patch]
         for i in local_idxs_matched], axis=1
    )


scaled_features_matched = {c: _subset_matched(arr) for c, arr in scaled_features_by_climate.items()}
print(f"Feature shape after subset: { {c: v.shape for c, v in scaled_features_matched.items()} }")


## ClimaX Architecture

**ClimaX tokenization scheme** (one token per variable × spatial-position pair):

```
token[v, p] = token_embeds[v](x[v, p])   # variable-specific scalar → D projection
            + var_embed[v]                 # learnable variable identity embedding
            + pos_embed[p]                 # learnable spatial position embedding
```

All `n_vars × n_patches` tokens are concatenated into a single sequence and processed
by a standard Transformer (multi-head self-attention + pre-norm FFN blocks).

After the Transformer, tokens are reshaped to `[B, n_vars, n_patches, D]` and
**mean-pooled over the variable dimension** → `[B, n_patches, D]`.

A **progressive compression** head then maps each position from `D=1024` to `latent_dim=64`
in two steps (`Linear(1024→256) → GELU → Linear(256→64)`), producing `[B, n_patches, 64]`.

**Attention pooling** over spatial positions (a learned `Linear(64→1)` + softmax)
produces the **global latent** `[B, 64]` — comparable to the 64-dim latent in CERA.

A **CERA-style predictor** (5 hidden FC layers, 128 units, LeakyReLU) maps the 64-dim
latent directly to all `n_patches` output values.

In [ ]:
# ============================================================
# ClimaX building blocks + two-part model
# ============================================================


class ClimaXMLP(nn.Module):
    # Feed-forward block with fc1/fc2 naming for pretrained-weight compatibility
    def __init__(self, in_features: int, hidden_features: int, drop: float = 0.0):
        super().__init__()
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class ClimaXAttention(nn.Module):
    # Multi-head self-attention with qkv/proj naming matching the ClimaX checkpoint
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, 3 * dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = self.attn_drop((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        return self.proj_drop(self.proj((attn @ v).transpose(1, 2).reshape(B, N, C)))


class ClimaXBlock(nn.Module):
    # Pre-norm Transformer block -- key names match the official ClimaX checkpoint
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = ClimaXAttention(dim, num_heads, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = ClimaXMLP(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ClimaXFeatureExtractor(nn.Module):
    # Frozen ClimaX-compatible feature extractor.
    #
    # Input  : x         [B, V, H, W]  H and W must be even (pad lon 7->8 before calling)
    #          pos_embed [B, P, D]     per-sample geographic pos_embed, P=(H/2)*(W/2)
    # Output : [B, P, D]              mean-pooled Transformer features
    #
    # Token: Conv2d_v(2x2 patch) + channel_embed[v] + pos_embed_geographic[p]
    # All V*P tokens go through the 8-block Transformer together.
    def __init__(self, n_vars, embed_dim=1024, depth=8, num_heads=16,
                 mlp_ratio=4.0, drop_rate=0.1, attn_drop_rate=0.0):
        super().__init__()
        self.n_vars    = n_vars
        self.embed_dim = embed_dim

        # Patch embedding -- Conv2d(1, D, kernel=2, stride=2) per variable.
        # Weights loaded from checkpoint key: token_embeds.{idx}.proj.{weight|bias}
        self.token_embeds = nn.ModuleList([
            nn.Conv2d(1, embed_dim, kernel_size=2, stride=2) for _ in range(n_vars)
        ])

        # Variable identity: [1, V, 1, D] -- broadcast over batch and patches.
        # Rows extracted from checkpoint key: channel_embed (1, 48, 1024)
        self.channel_embed = nn.Parameter(torch.zeros(1, n_vars, 1, embed_dim))
        nn.init.trunc_normal_(self.channel_embed, std=0.02)

        self.blocks = nn.ModuleList([
            ClimaXBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor, pos_embed: torch.Tensor) -> torch.Tensor:
        # x         : [B, V, H, W]   H and W even
        # pos_embed : [B, P, D]      P = (H/2)*(W/2)
        # returns   : [B, P, D]
        B, V, H, W = x.shape
        patch_tokens = []
        for v in range(V):
            tv = self.token_embeds[v](x[:, v:v+1])          # [B, D, H/2, W/2]
            patch_tokens.append(tv.flatten(2).transpose(1, 2))  # [B, P, D]
        tokens = torch.stack(patch_tokens, dim=1)            # [B, V, P, D]

        tokens = tokens + self.channel_embed                  # + [1, V, 1, D]
        tokens = tokens + pos_embed.unsqueeze(1)              # + [B, 1, P, D]

        P = tokens.shape[2]
        tokens = tokens.reshape(B, V * P, self.embed_dim)
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens).reshape(B, V, P, self.embed_dim)
        return tokens.mean(dim=1)                             # [B, P, D]


class ClimaXPredictorHead(nn.Module):
    # Prediction head -- the only trained part.
    #
    # Input  : agg [B, P, D]   precomputed backbone features (P=20, D=1024)
    # Output : (latent [B, latent_dim], pred [B, n_out])
    #
    # compress  : [B, P, 1024] -> [B, P, 256] -> [B, P, 64]
    # attn_pool : learned spatial soft-weighting -> [B, 64]  global latent
    # predictor : CERA-style 5-layer MLP -> [B, n_out=70]
    def __init__(self, embed_dim=1024, latent_dim=64, n_out=70, hidden_dim=128, n_hidden=5):
        super().__init__()
        self.compress  = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.GELU(), nn.Linear(256, latent_dim),
        )
        self.attn_pool = nn.Linear(latent_dim, 1, bias=False)
        layers, in_d = [], latent_dim
        for _ in range(n_hidden):
            layers += [nn.Linear(in_d, hidden_dim), nn.LeakyReLU(0.1)]
            in_d = hidden_dim
        layers.append(nn.Linear(in_d, n_out))
        self.predictor = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, agg: torch.Tensor):
        # agg: [B, P, D] -> (latent [B, latent_dim], pred [B, n_out])
        small  = self.compress(agg)
        w      = torch.softmax(self.attn_pool(small), dim=1)
        latent = (w * small).sum(dim=1)
        return latent, self.predictor(latent)


### ClimaX fine-tuning hyperparameters

In [ ]:
# ── ClimaX architecture -- must match the pretrained 5.625-degree checkpoint ──
climax_embed_dim        = 1024   # D=1024 (5.625deg checkpoint)
climax_depth            = 8      # 8 Transformer blocks
climax_num_heads        = 16     # 1024/16 = 64 head_dim
climax_mlp_ratio        = 4.0    # FFN hidden / embed
climax_drop_rate        = 0.1
climax_attn_drop_rate   = 0.0

# ── Latent / head ────────────────────────────────────────────────────────────
climax_latent_dim              = 64
climax_predictor_hidden_dim    = 128
climax_predictor_n_hidden_layers = 5

# ── Precomputed backbone features ────────────────────────────────────────────
CLIMAX_N_PATCHES_IN = (n_lat // 2) * ((n_lon + 1) // 2)  # = 5 * 4 = 20

# ── Training (head only -- no backbone in the loop) ──────────────────────────
climax_batch_size            = 512   # large: backbone precomputed, no gradient through it
climax_precompute_batch_size = 256   # batch for one-time backbone forward pass
climax_eval_batch_size       = 512
climax_n_epochs              = 100
climax_patience              = 15
climax_learning_rate         = 3e-4
climax_weight_decay          = 1e-5

# ── Checkpoint / cache paths ──────────────────────────────────────────────────
# climax_checkpoint_path stays on /glade/u/home (small file, needs to survive
# restarts reliably). climax_precomputed_dir lives on /glade/work: the
# backbone cache for all 5 climates is ~60 GB, which would eat most of the
# 100 GB /glade/u/home quota -- /glade/work has a 2 TB quota and is meant for
# this kind of large, regenerable scratch data.
climax_pretrained_path = Path(
    "/glade/u/home/tsalin/CMIP/model_evaluation/Baseline_ClimaX/climax_pretrained.ckpt"
)
climax_checkpoint_path = Path(
    "/glade/u/home/tsalin/CMIP/model_evaluation/Baseline_ClimaX/climax_head_checkpoint.pt"
)
climax_precomputed_dir = Path(
    "/glade/work/tsalin/CMIP/model_evaluation/Baseline_ClimaX/precomputed_features"
)
climax_checkpoint_freq = 1

# Number of precompute batches between two disk flushes (checkpoint
# granularity for the backbone precompute -- see cell below).
climax_precompute_flush_every_n_batches = 30

In [ ]:
# ── Download ClimaX pretrained checkpoint ────────────────────────────────────
# Source : https://huggingface.co/tungnd/climax (official Microsoft ClimaX weights)
# File   : 5.625deg.ckpt  (~880 MB, PyTorch Lightning format)
# Weights transferred to our model: 8 transformer blocks + final LayerNorm.
# All other parameters (token_embeds, var_embed, pos_embed, head) are randomly
# initialised and trained from scratch during fine-tuning.

CLIMAX_CHECKPOINT_URL = (
    "https://huggingface.co/tungnd/climax/resolve/main/5.625deg.ckpt"
)

def _download_climax_checkpoint_if_needed(path):
    """Download the ClimaX pretrained checkpoint if not already on disk."""
    path = Path(path)
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f"[ClimaX] Checkpoint already present : {path}  ({size_mb:.0f} MB)")
        return True

    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ClimaX] Pretrained checkpoint not found — downloading (~880 MB) ...")
    print(f"         URL  : {CLIMAX_CHECKPOINT_URL}")
    print(f"         Dest : {path}")
    print("         (If this cell times out on a compute node without internet,")
    print("          run the wget command below from the login node instead.)")
    try:
        torch.hub.download_url_to_file(CLIMAX_CHECKPOINT_URL, str(path), progress=True)
        size_mb = path.stat().st_size / 1e6
        print(f"[ClimaX] Download complete ({size_mb:.0f} MB).")
        return True
    except Exception as exc:
        print(f"[ClimaX] Automatic download failed: {exc}")
        print()
        print("  Download manually from the login node and then re-run this cell:")
        print(f"  wget '{CLIMAX_CHECKPOINT_URL}' \\")
        print(f"       -O '{path}'")
        return False

_download_climax_checkpoint_if_needed(climax_pretrained_path)

In [ ]:
def _load_pretrained_climax_weights(model, ckpt_path, matched_climax_idxs):
    # Transfer pretrained ClimaX weights into a ClimaXFeatureExtractor.
    #
    # Transferred tensors:
    #   blocks.{0..7}  : 96 Transformer tensors (norms, qkv, proj, mlp.fc1/fc2)
    #   norm           : 2 final-LayerNorm tensors
    #   channel_embed  : rows matched_climax_idxs from ckpt (1,48,1024) -> our (1,V,1,1024)
    #   token_embeds.v : Conv2d(1,1024,2,2) from ckpt token_embeds.{idx}.proj.*
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print("[ClimaX] Checkpoint not found. Using random init.")
        return False
    print(f"[ClimaX] Loading pretrained weights from {ckpt_path} ...")
    raw   = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = raw.get("state_dict", raw)
    state = {(k[4:] if k.startswith("net.") else k): v for k, v in state.items()}
    our   = model.state_dict()
    loaded = []

    # Transformer blocks
    for i in range(len(model.blocks)):
        for sfx in ("norm1.weight", "norm1.bias", "attn.qkv.weight", "attn.qkv.bias",
                     "attn.proj.weight", "attn.proj.bias", "norm2.weight", "norm2.bias",
                     "mlp.fc1.weight", "mlp.fc1.bias", "mlp.fc2.weight", "mlp.fc2.bias"):
            k = f"blocks.{i}.{sfx}"
            if k in our and k in state and our[k].shape == state[k].shape:
                our[k].copy_(state[k])
                loaded.append(k)

    # Final LayerNorm
    for k in ("norm.weight", "norm.bias"):
        if k in our and k in state and our[k].shape == state[k].shape:
            our[k].copy_(state[k])
            loaded.append(k)

    # channel_embed: ckpt (1,48,1024) -> our (1, n_matched, 1, 1024)
    ckpt_chan = state["channel_embed"]
    for our_v, cidx in enumerate(matched_climax_idxs):
        our["channel_embed"][0, our_v, 0, :].copy_(ckpt_chan[0, cidx, :])
    loaded.append("channel_embed")

    # token_embeds: ckpt has extra .proj. level vs our nn.Conv2d keys
    n_tok = 0
    for our_v, cidx in enumerate(matched_climax_idxs):
        ck_w = f"token_embeds.{cidx}.proj.weight"
        ck_b = f"token_embeds.{cidx}.proj.bias"
        ou_w = f"token_embeds.{our_v}.weight"
        ou_b = f"token_embeds.{our_v}.bias"
        if ck_w in state and our[ou_w].shape == state[ck_w].shape:
            our[ou_w].copy_(state[ck_w])
            our[ou_b].copy_(state[ck_b])
            n_tok += 1

    model.load_state_dict(our)
    n_blk = sum(1 for k in loaded if isinstance(k, str) and k.startswith("blocks."))
    print(f"[ClimaX] Transferred: {n_blk} block tensors, 2 norm, channel_embed, {n_tok} token_embeds")
    return True


In [ ]:
# ── Instantiate ClimaXFeatureExtractor and load pretrained weights ────────────
feature_extractor = ClimaXFeatureExtractor(
    n_vars         = n_matched,
    embed_dim      = climax_embed_dim,
    depth          = climax_depth,
    num_heads      = climax_num_heads,
    mlp_ratio      = climax_mlp_ratio,
    drop_rate      = climax_drop_rate,
    attn_drop_rate = climax_attn_drop_rate,
).to(device)

_load_pretrained_climax_weights(feature_extractor, climax_pretrained_path, matched_climax_idxs)

for p in feature_extractor.parameters():
    p.requires_grad = False
feature_extractor.eval()

n_total = sum(p.numel() for p in feature_extractor.parameters())
print(f"ClimaXFeatureExtractor : {n_total:,} params (all frozen)")
print(f"Input  : [B, {n_matched}, {n_lat}, {n_lon}] -> padded [B, {n_matched}, {n_lat}, {n_lon + 1}]")
print(f"Tokens : {CLIMAX_N_PATCHES_IN} patch positions/sample ({n_lat // 2} lat x {(n_lon + 1) // 2} lon)")


In [ ]:
# ── Precompute geographic pos_embed for all 246 patch_ids ────────────────────
#
# The ClimaX pretrained pos_embed (1, 512, 1024) encodes the 16x32 patch
# positions of the global 5.625 deg ERA5 grid. For each user patch_id we know
# the exact lat/lon of its 20 patch centers and bilinearly interpolate the
# pretrained pos_embed there, giving a sample-specific position encoding that
# uses ClimaX's pretrained spatial knowledge without changing the input size.

from scipy.interpolate import RegularGridInterpolator

# ClimaX 5.625 deg global grid
_climax_grid_lats = 90.0 - 5.625 * np.arange(32)   # [90, 84.375, ..., -84.375]  N->S
_climax_grid_lons = 5.625 * np.arange(64)            # [0, 5.625, ..., 354.375]    W->E

# Patch centers after 2x2 Conv2d (mean of each consecutive pair)
_cpatch_lats = (_climax_grid_lats[0::2] + _climax_grid_lats[1::2]) / 2  # 16 vals descending
_cpatch_lons = (_climax_grid_lons[0::2] + _climax_grid_lons[1::2]) / 2  # 32 vals ascending

# Load pretrained pos_embed (1, 512, 1024) -> reshape to (16, 32, 1024)
_raw = torch.load(climax_pretrained_path, map_location="cpu", weights_only=False)
_st  = _raw.get("state_dict", _raw)
_st  = {(k[4:] if k.startswith("net.") else k): v for k, v in _st.items()}
_pos_grid = _st["pos_embed"].squeeze(0).reshape(16, 32, climax_embed_dim).numpy()
del _raw, _st

# RegularGridInterpolator requires strictly increasing first axis -> flip lat
_interp = RegularGridInterpolator(
    (_cpatch_lats[::-1].copy(), _cpatch_lons),
    _pos_grid[::-1].copy(),
    method="linear", bounds_error=False, fill_value=None,
)

_N_PH = n_lat // 2        # = 5  lat patch rows  (10 / 2)
_N_PW = (n_lon + 1) // 2  # = 4  lon patch cols  (8 / 2, after replication padding)


def _geographic_pos_embed(patch_id: int) -> torch.Tensor:
    # Bilinearly interpolate the ClimaX pretrained pos_embed at the 20 geographic
    # positions of a patch's 2x2 patch centers.
    # Returns [20, embed_dim]. Row-major order matches Conv2d flatten:
    # (lat_row=0, lon_col=0) ... (lat_row=4, lon_col=3).
    row   = patch_catalog[patch_catalog["patch_id"] == patch_id].iloc[0]
    glats = np.linspace(row["lat_start"], row["lat_stop"], n_lat)   # 10 lat values
    glons = np.linspace(row["lon_start"], row["lon_stop"], n_lon)   # 7 lon values
    glons = np.append(glons, glons[-1])                              # replicate -> 8
    plats = (glats[0::2] + glats[1::2]) / 2                         # 5 patch lat centers
    plons = (glons[0::2] + glons[1::2]) / 2                         # 4 patch lon centers
    lat_g, lon_g = np.meshgrid(plats, plons, indexing="ij")         # (5, 4)
    query = np.stack([lat_g.flatten(), lon_g.flatten()], axis=1)    # (20, 2)
    return torch.from_numpy(_interp(query).astype(np.float32))      # (20, 1024)


pos_embed_by_patch_id = {
    int(pid): _geographic_pos_embed(int(pid))
    for pid in sorted(patch_catalog["patch_id"].unique())
}
print(f"Precomputed geographic pos_embed for {len(pos_embed_by_patch_id)} patches.")
print(f"Shape per patch  : {next(iter(pos_embed_by_patch_id.values())).shape}")
print(f"ClimaX lat range : {_cpatch_lats[-1]:.2f} deg to {_cpatch_lats[0]:.2f} deg")
print(f"ClimaX lon range : {_cpatch_lons[0]:.2f} deg to {_cpatch_lons[-1]:.2f} deg")


In [ ]:
# ── Precompute backbone features for all samples (one-time, cached to disk) ───
# All embeddings and the Transformer backbone are fully frozen -> the backbone
# output for each sample is deterministic -> computed once, cached, reused every epoch.
#
# Checkpointed every `climax_precompute_flush_every_n_batches` batches: results
# are written into an on-disk memmap (`backbone_{climate}.tmp.npy`) as they are
# computed, with a small sidecar progress marker (`backbone_{climate}.tmp.progress`)
# recording how many rows are safely flushed to disk. If the job is killed
# mid-climate, the next run resumes from the last flushed batch instead of
# recomputing the whole climate. The climate is only considered "done" once
# the tmp file is atomically renamed to `backbone_{climate}.npy` -- that is the
# only file the rest of the notebook (and cell 40's cleanup) checks for.

climax_precomputed_dir.mkdir(parents=True, exist_ok=True)


def _precompute_backbone_features(climate: str, force: bool = False) -> np.ndarray:
    # Run all samples of a climate through the frozen ClimaXFeatureExtractor.
    # Returns [n, CLIMAX_N_PATCHES_IN, embed_dim] = [n, 20, 1024] (float16, mmap'd).
    cache_path    = climax_precomputed_dir / f"backbone_{climate}.npy"
    tmp_path      = climax_precomputed_dir / f"backbone_{climate}.tmp.npy"
    progress_path = climax_precomputed_dir / f"backbone_{climate}.tmp.progress"

    if cache_path.exists() and not force:
        data = np.load(str(cache_path), mmap_mode='r')
        print(f"[Precompute] Loaded {climate}: {data.shape}  dtype={data.dtype}")
        log_progress(f"[precompute] {climate}: loaded from cache ({data.shape})")
        return data

    X_flat = scaled_features_matched[climate]
    meta   = metadata_by_climate[climate]
    n      = X_flat.shape[0]
    shape  = (n, CLIMAX_N_PATCHES_IN, climax_embed_dim)

    if tmp_path.exists() and progress_path.exists():
        n_done = int(progress_path.read_text().strip())
        result = np.lib.format.open_memmap(str(tmp_path), mode='r+', dtype=np.float16, shape=shape)
        print(f"[Precompute] Resuming {climate} from row {n_done}/{n}")
        log_progress(f"[precompute] {climate}: resuming from row {n_done}/{n}")
    else:
        n_done = 0
        result = np.lib.format.open_memmap(str(tmp_path), mode='w+', dtype=np.float16, shape=shape)
        print(f"[Precompute] Computing backbone features for {climate} ...")
        log_progress(f"[precompute] {climate}: starting from scratch (n={n})")

    batches_since_flush = 0
    with torch.inference_mode():
        for start in range(n_done, n, climax_precompute_batch_size):
            end = min(start + climax_precompute_batch_size, n)
            bs  = end - start
            xb  = torch.from_numpy(
                X_flat[start:end].reshape(bs, n_matched, n_lat, n_lon).astype(np.float32)
            )
            # Pad lon 7 -> 8: replicate last column so Conv2d stride=2 divides evenly
            xb   = torch.cat([xb, xb[:, :, :, -1:]], dim=3)          # [bs, V, 10, 8]
            pids = meta["patch_id"].values[start:end]
            pos  = torch.stack(
                [pos_embed_by_patch_id[int(p)] for p in pids]
            )                                                           # [bs, 20, 1024]
            feats = feature_extractor(xb.to(device), pos.to(device))  # [bs, 20, 1024]

            result[start:end] = feats.cpu().half().numpy()
            batches_since_flush += 1

            if batches_since_flush >= climax_precompute_flush_every_n_batches:
                result.flush()
                progress_path.write_text(str(end))
                log_progress(f"[precompute] {climate}: {end}/{n} ({100 * end / n:.1f}%)")
                batches_since_flush = 0

    result.flush()                # flush the remainder (< flush_every_n batches)
    progress_path.write_text(str(n))
    del result
    os.replace(str(tmp_path), str(cache_path))   # atomic commit -> climate marked "done"
    progress_path.unlink(missing_ok=True)

    print(f"[Precompute] Saved {climate}: {shape}  dtype=float16  -> {cache_path}")
    log_progress(f"[precompute] {climate}: done, saved to {cache_path}")
    return np.load(str(cache_path), mmap_mode='r')


precomputed_features = {c: _precompute_backbone_features(c) for c in climate_order}
print("\nBackbone feature shapes:")
for c, f in precomputed_features.items():
    print(f"  {c}: {f.shape}  dtype={f.dtype}")

In [ ]:
def train_climax_head(
    n_epochs=100, lr=3e-4, batch_size=512, weight_decay=1e-5,
    patience=15, checkpoint_path=None, checkpoint_freq=1,
):
    # Train only the ClimaXPredictorHead on precomputed backbone features.
    # No gradient flows through the backbone -- only compress+attn_pool+predictor train.
    head = ClimaXPredictorHead(
        embed_dim  = climax_embed_dim,
        latent_dim = climax_latent_dim,
        n_out      = grid_points_per_patch,
        hidden_dim = climax_predictor_hidden_dim,
        n_hidden   = climax_predictor_n_hidden_layers,
    ).to(device)
    print(f"ClimaXPredictorHead: {sum(p.numel() for p in head.parameters()):,} trainable params")

    train_idx = ae_split_indices["historical"]["train"]
    val_idx   = ae_split_indices["historical"]["val"]

    # Keep float16 tensors for the DataLoader; cast to float32 only when moving to device.
    F_tr = torch.from_numpy(np.array(precomputed_features["historical"][train_idx]))
    y_tr = torch.from_numpy(label_variable_by_climate["historical"][train_idx])
    F_va = torch.from_numpy(np.array(precomputed_features["historical"][val_idx]))
    y_va = torch.from_numpy(label_variable_by_climate["historical"][val_idx])

    train_ld = DataLoader(TensorDataset(F_tr, y_tr), batch_size=batch_size,
                          shuffle=True,  drop_last=True,  pin_memory=(device.type == "cuda"))
    val_ld   = DataLoader(TensorDataset(F_va, y_va), batch_size=batch_size,
                          shuffle=False, drop_last=False, pin_memory=(device.type == "cuda"))

    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    mse = nn.MSELoss()
    best_val, best_epoch, wait, best_state, history = np.inf, -1, 0, None, []
    start_epoch = 1

    if checkpoint_path and Path(checkpoint_path).exists():
        ck = torch.load(checkpoint_path, map_location=device)
        head.load_state_dict(ck["model_state"])
        opt.load_state_dict(ck["optimizer_state"])
        start_epoch = ck["epoch"] + 1
        best_val    = ck["best_val"]
        best_epoch  = ck["best_epoch"]
        wait        = ck["patience_counter"]
        best_state  = ck["best_model_state"]
        history     = ck["history"]
        print(f"[CHECKPOINT] Resuming from epoch {start_epoch} (best val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        head.train()
        tl = []
        for Fb, yb in train_ld:
            Fb, yb = Fb.to(device, dtype=torch.float32), yb.to(device)
            opt.zero_grad()
            _, pred = head(Fb)
            loss = mse(pred, yb)
            loss.backward()
            opt.step()
            tl.append(loss.item())

        head.eval()
        vl = []
        with torch.inference_mode():
            for Fb, yb in val_ld:
                _, pred = head(Fb.to(device, dtype=torch.float32))
                vl.append(mse(pred, yb.to(device)).item())

        tl_m, vl_m = float(np.mean(tl)), float(np.mean(vl))
        history.append({"epoch": epoch, "train_loss": tl_m, "val_loss": vl_m})
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:4d}/{n_epochs} | train={tl_m:.4f} | val={vl_m:.4f}")
            log_progress(f"[train] epoch {epoch}/{n_epochs} train={tl_m:.4f} val={vl_m:.4f}")

        def _save(p):
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in head.state_dict().items()},
                "optimizer_state": opt.state_dict(),
                "best_val": best_val, "best_epoch": best_epoch,
                "patience_counter": wait,
                "best_model_state": best_state, "history": history,
            }, p)

        if checkpoint_path and epoch % checkpoint_freq == 0:
            _save(checkpoint_path)

        if vl_m < best_val:
            best_val, best_epoch, best_state, wait = (
                vl_m, epoch,
                {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, 0)
        else:
            wait += 1
            if wait >= patience:
                if checkpoint_path:
                    _save(checkpoint_path)
                print(f"[EarlyStopping] Stopped at epoch {epoch} "
                      f"(best: {best_epoch}, val={best_val:.6f})")
                log_progress(f"[train] early stopping at epoch {epoch} (best: {best_epoch}, val={best_val:.6f})")
                break

    if best_state:
        head.load_state_dict(best_state)
    return head, pd.DataFrame(history), best_epoch


### Fine-tuning ClimaX

In [ ]:
Path(climax_checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

climax_head, climax_history_df, climax_best_epoch = train_climax_head(
    n_epochs        = climax_n_epochs,
    lr              = climax_learning_rate,
    batch_size      = climax_batch_size,
    weight_decay    = climax_weight_decay,
    patience        = climax_patience,
    checkpoint_path = climax_checkpoint_path,
    checkpoint_freq = climax_checkpoint_freq,
)

print(f'Best epoch : {climax_best_epoch}')
display(climax_history_df.tail(10))


### Evaluation on test sets — EXTRACTION

We evaluate prediction quality on all climate test sets (historical + all SSPs).
Reminder: the model was fine-tuned only on historical train; validation used historical val.

The cell also extracts **latent representations** (global mean-pool of the Transformer
output before the head) for all test samples.

**Note**: the denormalized precipitation is in **mm/day**.

In [ ]:
def _climax_predict_and_extract_latents_batched(
    head: ClimaXPredictorHead,
    features: np.ndarray,
    batch_size: int,
) -> tuple:
    # features: [n, 20, 1024]  precomputed backbone features (float16 or float32)
    # returns (pred [n, G], latent [n, latent_dim])
    n   = features.shape[0]
    F_t = torch.from_numpy(np.ascontiguousarray(features))   # keeps original dtype (float16)
    ds  = torch.utils.data.TensorDataset(F_t)
    ld  = torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=False, drop_last=False,
        pin_memory=(device.type == 'cuda'),
    )
    preds, latents = [], []
    head.eval()
    with torch.inference_mode():
        for (Fb,) in ld:
            lat, pred = head(Fb.to(device, dtype=torch.float32))
            preds.append(pred.cpu().numpy())
            latents.append(lat.cpu().numpy())
    return np.concatenate(preds, axis=0), np.concatenate(latents, axis=0)


# ── Build quality payload ───────────────────────────────────────────────
component_order = {'prediction': 0}


def _build_meta_df(component, climate, metadata):
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=['scenario'], errors='ignore')
    df.insert(0, 'experiment',      'CMIP_variable_exp5_baseline_ClimaX')
    df.insert(1, 'component',       component)
    df.insert(2, 'component_order', component_order[component])
    df.insert(3, 'scenario',        climate)
    df.insert(4, 'scenario_order',  int(climate_order.index(climate)))
    df.insert(5, 'sample_idx',      range(len(df)))
    return df


prediction_value_names = [
    f'{variable}@point_{p}' for p in range(grid_points_per_patch)
]

meta_dfs_pred     = []
truth_arrays_pred = []
pred_arrays_pred  = []

climax_latent_by_climate               = {}
climax_latent_test_metadata_by_climate = {}

for climate in climate_order:
    idx      = ae_split_indices[climate]['test']
    feats    = precomputed_features[climate][idx]   # [n, 20, 1024]  float16
    y_true   = np.asarray(label_variable_by_climate[climate][idx], dtype=np.float32)
    metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    y_hat, z_np = _climax_predict_and_extract_latents_batched(
        climax_head, feats, climax_eval_batch_size,
    )

    climax_latent_by_climate[climate]               = z_np
    climax_latent_test_metadata_by_climate[climate] = metadata

    y_true_phys = denormalize_label_variable(y_true)
    y_hat_phys  = denormalize_label_variable(y_hat)

    meta_dfs_pred.append(_build_meta_df('prediction', climate, metadata))
    truth_arrays_pred.append(y_true_phys.astype(np.float32))
    pred_arrays_pred.append(y_hat_phys.astype(np.float32))

climax_quality_payload = {
    'meta_prediction':        pd.concat(meta_dfs_pred,   ignore_index=True),
    'truth_prediction':       np.concatenate(truth_arrays_pred, axis=0),
    'pred_prediction':        np.concatenate(pred_arrays_pred,  axis=0),
    'prediction_value_names': prediction_value_names,
}

print('ClimaX quality payload -- shapes:')
print(f"  meta_prediction  : {climax_quality_payload['meta_prediction'].shape}")
print(f"  truth_prediction : {climax_quality_payload['truth_prediction'].shape}")
print(f"  pred_prediction  : {climax_quality_payload['pred_prediction'].shape}")
print()
print(climax_quality_payload['meta_prediction'].groupby('scenario').size().rename('n_samples'))
display(climax_quality_payload['meta_prediction'].head())


### Latent representations

The latent representations were already extracted during the evaluation cell above.

**Extraction point**: after the progressive compression (`Linear(1024→256)→GELU→Linear(256→64)`)
and attention pooling over spatial positions — i.e. the **64-dim global latent** before
the CERA-style predictor head. This is directly comparable to the 64-dim latent from CERA
and other baselines (`ae_latent_dim = 64`).

Stored in `climax_latent_by_climate` and `climax_latent_test_metadata_by_climate`.

In [ ]:
if "climax_latent_by_climate" not in globals() or "climax_latent_test_metadata_by_climate" not in globals():
    raise RuntimeError(
        "Run the evaluation cell before this latent check cell."
    )

for climate, z in climax_latent_by_climate.items():
    print(f"  {climate:12s}: latent shape = {z.shape}")

Delete the checkpoint once the notebook has finished successfully

In [ ]:
if climax_checkpoint_path.exists():
    climax_checkpoint_path.unlink()
    print(f"[CHECKPOINT] Checkpoint deleted: {climax_checkpoint_path}")
else:
    print("[CHECKPOINT] No checkpoint to delete.")

_deleted = sorted(climax_precomputed_dir.glob("backbone_*"))  # catches final .npy + leftover .tmp.npy/.tmp.progress
for _f in _deleted:
    _f.unlink()
if _deleted:
    print(f"[PRECOMPUTE] Features deleted ({len(_deleted)}): {[f.name for f in _deleted]}")
else:
    print("[PRECOMPUTE] No precomputed feature to delete.")
